### MODFLOW 6 Output Processors

This set of codes was designed to process MODFLOW 6 output files (recarrays and txts) to generate GIS-compatible shapfiles and rasters.

---

### Authorship

**Author:** Kaden McCulloch (https://www.linkedin.com/in/kfmcculloch)
<br>
**Date:** 2026-01-30
<br>
**Version:** 1.0

---

### Acknowledgments
Special thanks to Mike Fienen (USGS) for sharing his model to support the development of this code.

---

### Contact
For questions, please reach out to [kaden.mcculloch@gmail.com](mailto:kaden.mcculloch@gmail.com)


In [11]:
import flopy as fp
import geopandas as gp 
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from pathlib import Path
import numpy as np
import pyemu
import platform, shutil
import re
import pandas as pd
from io import StringIO
from flopy.export.shapefile_utils import recarray2shp
from flopy.utils.geometry import Polygon


### set paths to information we will need

In [4]:
#### Base directory for runs
parent_run_path = Path("../../LPR_pycap_opt/pycap_runs")

#### depletion potential calculations directory
base_run_path = parent_run_path / "pycap_base"

#### modflow base path
modflow_run_path = Path("/workspaces/modflow")

#### modflow depletion potential calculations directory
modflow_dp_path = Path('../steady_state_mf6_DP')

### read in the MODFLOW-6 version of the steady-state Little Plover River Model

In [5]:
sim = fp.mf6.MFSimulation.load(sim_ws = '../steady_state_mf6')
m = sim.get_model()

loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package sfr...
    loading package ic...
    loading package npf...
    loading package oc...
    loading package rch...
    loading package chd...
    loading package drn...
    loading package wel...
  loading solution package lpr_ss_gwf...


In [6]:
modelgrid = fp.discretization.StructuredGrid(xoff=1795400*.3048,  
                                             yoff=1424400*.3048,
                                             botm=m.dis.botm.array,
                                             delr = m.dis.delr.array*.3048,
                                             delc = m.dis.delc.array*.3048,
                                                
                                             crs=3071)


In [7]:
idomain = [[int(i) for i in j.split()] for j in open('../steady_state_mf6/lpr_ss_gwf.dis_idomain_layer1.txt','r').readlines()]
idomain = np.array([k for j in idomain for k in j]).reshape(m.dis.nrow.array,m.dis.ncol.array)

### recarray to shapefile

In [8]:
### chd
chds = pd.read_csv('../steady_state_mf6/lpr_ss_gwf.chd_stress_period_data_1.txt', sep=r'\s+', header=None, names = ['k','i','j','head'])
for cc in ['k','i','j']:
    chds[cc]-=1
chds

chd_shp = [Polygon(modelgrid.get_cell_vertices(row, col)) for row, col in zip(chds.i, chds.j)]
recarray2shp(chds.to_records(), geoms=chd_shp, shpname='../steady_state_mf6/chds.shp', crs=modelgrid.crs)
chd_map = gp.read_file('../steady_state_mf6/chds.shp')
chd_map = chd_map.set_crs(modelgrid.crs)

In [9]:
### wel
wels_df = pd.read_csv(
        '../steady_state_mf6/lpr_ss_gwf.wel_stress_period_data_1.txt',
            sep=r'\s+', header=None, names=['k','i','j','Q']
            )
for cc in ['k','i','j']: wels_df[cc] = wels_df[cc] - 1

geoms = [Polygon(modelgrid.get_cell_vertices(int(r), int(c))) for r, c in zip(wels_df.i, wels_df.j)]

recarray2shp(wels_df.to_records(index=False), geoms=geoms, shpname='../steady_state_mf6/wels.shp', crs=modelgrid.crs)

wel_map = gp.read_file('../steady_state_mf6/wels.shp').set_crs(modelgrid.crs)
wel_map.head()

,k,i,j,Q,geometry
0,0,638,975,-1483.065800,"POLYGON ((576955.92 442142.88, 576986.4 442142.88, 576986.4 442112.4, 576955.92 442112.4, 576955..."
1,1,638,975,-7071.723140,"POLYGON ((576955.92 442142.88, 576986.4 442142.88, 576986.4 442112.4, 576955.92 442112.4, 576955..."
2,0,751,93,-8784.499020,"POLYGON ((550072.56 438698.64, 550103.04 438698.64, 550103.04 438668.16, 550072.56 438668.16, 55..."
3,1,751,93,-8431.159180,"POLYGON ((550072.56 438698.64, 550103.04 438698.64, 550103.04 438668.16, 550072.56 438668.16, 55..."
4,1,751,93,-338.600769,"POLYGON ((550072.56 438698.64, 550103.04 438698.64, 550103.04 438668.16, 550072.56 438668.16, 55..."


In [12]:
### sfr

fn = '../steady_state_mf6/lpr_ss.sfr'

with open(fn, 'r') as f:
    lines = f.readlines()

# find the line index that begins with the "#rno" header (case-insensitive)
header_idx = None
for idx, ln in enumerate(lines):
    if ln.lstrip().lower().startswith('#rno'):
        header_idx = idx
        break

if header_idx is None:
    raise RuntimeError("Header '#rno ...' not found in file")

# collect data lines after the header until 'END Packagedata'
data_lines = []
for ln in lines[header_idx + 1:]:
    if ln.strip().upper() == 'END PACKAGEDATA':
        break
    # accept lines which start with a digit (rno) and skip blank/comment lines
    if re.match(r'^\s*\d+', ln):
        data_lines.append(ln)

if not data_lines:
    raise RuntimeError("No numeric data lines found after the '#rno' header")

# create a DataFrame from the collected lines and keep first 4 numeric columns (rno,k,i,j)
data_str = ''.join(data_lines)
df = pd.read_csv(StringIO(data_str), sep=r'\s+', header=None, engine='python')

# ensure there are at least 4 columns, then select columns 1,2,3 as k,i,j
if df.shape[1] < 4:
    raise RuntimeError("Parsed data has fewer than 4 columns; file format may differ")

sfrs_df = df.iloc[:, 1:4].copy()
sfrs_df.columns = ['k', 'i', 'j']

# convert to integers and zero-based indices
sfrs_df = sfrs_df.astype(int) - 1

print(sfrs_df.head())

geoms = [Polygon(modelgrid.get_cell_vertices(int(r), int(c))) for r, c in zip(sfrs_df.i, sfrs_df.j)]

recarray2shp(sfrs_df.to_records(index=False), geoms=geoms, shpname='../steady_state_mf6/sfrs.shp', crs=modelgrid.crs)

sfr_map = gp.read_file('../steady_state_mf6/sfrs.shp').set_crs(modelgrid.crs)
sfr_map.head()

   k    i    j
0  0  569  457
1  0  569  456
2  0  568  456
3  0  568  455
4  0  568  454


,k,i,j,geometry
0,0,569,457,"POLYGON ((561167.28 444246, 561197.76 444246, 561197.76 444215.52, 561167.28 444215.52, 561167.2..."
1,0,569,456,"POLYGON ((561136.8 444246, 561167.28 444246, 561167.28 444215.52, 561136.8 444215.52, 561136.8 4..."
2,0,568,456,"POLYGON ((561136.8 444276.48, 561167.28 444276.48, 561167.28 444246, 561136.8 444246, 561136.8 4..."
3,0,568,455,"POLYGON ((561106.32 444276.48, 561136.8 444276.48, 561136.8 444246, 561106.32 444246, 561106.32 ..."
4,0,568,454,"POLYGON ((561075.84 444276.48, 561106.32 444276.48, 561106.32 444246, 561075.84 444246, 561075.8..."


### txt to raster

In [ ]:
### top elevation

# 1. Load idomain and top elevation from text files
def load_mf6_text_file(path, nrow, ncol):
    with open(path, 'r') as f:
        data = f.read().split()
    return np.array(data, dtype=float).reshape(nrow, ncol)

nrow, ncol = m.dis.nrow.array, m.dis.ncol.array

# Load custom arrays
idomain_layer1 = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.dis_idomain_layer1.txt', nrow, ncol).astype(int)
elev_data = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.dis_top.txt', nrow, ncol)

# 2. Define the StructuredGrid with your specific offsets and CRS
modelgrid = fp.discretization.StructuredGrid(
    delr=m.dis.delr.array * 0.3048,
    delc=m.dis.delc.array * 0.3048,
    xoff=1795400 * 0.3048,
    yoff=1424400 * 0.3048,
    top=elev_data * 0.3048,       # Use the custom loaded top data
    botm=m.dis.botm.array * 0.3048,
    idomain=idomain_layer1,      # Use the custom loaded idomain
    crs="EPSG:3071"              # Wisconsin Transverse Mercator
)

# 3. Final Export to GeoTIFF
# Scaling the array by 0.3048 ensures the vertical Z-values in the TIF are in meters
fp.export.utils.export_array(
    modelgrid, 
    "lpr_ss_top_elevation.tif", 
    elev_data, 
    nodata=-999
)

/opt/conda/envs/hwrs564b/lib/python3.11/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems


In [ ]:
### domain

# 1. Load idomain and domain information from text files
def load_mf6_text_file(path, nrow, ncol):
    with open(path, 'r') as f:
        data = f.read().split()
    return np.array(data, dtype=float).reshape(nrow, ncol)

nrow, ncol = m.dis.nrow.array, m.dis.ncol.array

# Load custom arrays
idomain_layer1 = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.dis_idomain_layer1.txt', nrow, ncol).astype(int)
domain_data = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.dis_idomain_layer1.txt', nrow, ncol)

# 2. Define the StructuredGrid (Geometry ONLY)
modelgrid = fp.discretization.StructuredGrid(
    delr=m.dis.delr.array * 0.3048,
    delc=m.dis.delc.array * 0.3048,
    xoff=1795400 * 0.3048,
    yoff=1424400 * 0.3048,
    top=m.dis.top.array * 0.3048, # Use model top for geometry
    botm=m.dis.botm.array * 0.3048,
    idomain=idomain_layer1,
    crs="EPSG:3071"
)

# 3. Export the data to GeoTIFF
# Use the modelgrid for spatial context, but domain_data for the values
fp.export.utils.export_array(
    modelgrid, 
    "lpr_ss_gwf_idomain_layer1.tif", 
    domain_data, # This is where the data goes
    nodata=-999
)

/opt/conda/envs/hwrs564b/lib/python3.11/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems


In [ ]:
### horizontal k

# 1. Load idomain and top elevation from text files
def load_mf6_text_file(path, nrow, ncol):
    with open(path, 'r') as f:
        data = f.read().split()
    return np.array(data, dtype=float).reshape(nrow, ncol)

nrow, ncol = m.dis.nrow.array, m.dis.ncol.array

# Load custom arrays
idomain_layer3 = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.dis_idomain_layer3.txt', nrow, ncol).astype(int)
hk_data = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.npf_k_layer3.txt', nrow, ncol)

# 2. Define the StructuredGrid (Geometry ONLY)
modelgrid = fp.discretization.StructuredGrid(
    delr=m.dis.delr.array * 0.3048,
    delc=m.dis.delc.array * 0.3048,
    xoff=1795400 * 0.3048,
    yoff=1424400 * 0.3048,
    top=m.dis.top.array * 0.3048, # Use model top for geometry
    botm=m.dis.botm.array * 0.3048,
    idomain=idomain_layer3,
    crs="EPSG:3071"
)

# 3. Export the Hydraulic Conductivity to GeoTIFF
# Use the modelgrid for spatial context, but hk_data for the values
fp.export.utils.export_array(
    modelgrid, 
    "lpr_ss_gwf_hk_layer3.tif", 
    hk_data, # This is where the data goes
    nodata=-999
)

In [ ]:
### initial heads

# 1. Load idomain and initial heads from text files
def load_mf6_text_file(path, nrow, ncol):
    with open(path, 'r') as f:
        data = f.read().split()
    return np.array(data, dtype=float).reshape(nrow, ncol)

nrow, ncol = m.dis.nrow.array, m.dis.ncol.array

# Load custom arrays
idomain_layer3 = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.dis_idomain_layer3.txt', nrow, ncol).astype(int)
ic_data = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.ic_strt_layer3.txt', nrow, ncol)

# 2. Define the StructuredGrid (Geometry ONLY)
modelgrid = fp.discretization.StructuredGrid(
    delr=m.dis.delr.array * 0.3048,
    delc=m.dis.delc.array * 0.3048,
    xoff=1795400 * 0.3048,
    yoff=1424400 * 0.3048,
    top=m.dis.top.array * 0.3048, # Use model top for geometry
    botm=m.dis.botm.array * 0.3048,
    idomain=idomain_layer3,
    crs="EPSG:3071"
)

# 3. Export the initial heads to GeoTIFF
# Use the modelgrid for spatial context, but ic_data for the values
fp.export.utils.export_array(
    modelgrid, 
    "lpr_ss_gwf.ic_strt_layer3.tif", 
    ic_data, # This is where the data goes
    nodata=-999
)

/opt/conda/envs/hwrs564b/lib/python3.11/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems


In [14]:
### recharge

# 1. Load idomain and recharge from text files
def load_mf6_text_file(path, nrow, ncol):
    with open(path, 'r') as f:
        data = f.read().split()
    return np.array(data, dtype=float).reshape(nrow, ncol)

nrow, ncol = m.dis.nrow.array, m.dis.ncol.array

# Load custom arrays
idomain_layer1 = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.dis_idomain_layer1.txt', nrow, ncol).astype(int)
rcha_data = load_mf6_text_file('../steady_state_mf6/lpr_ss_gwf.rcha_recharge_1.txt', nrow, ncol)

# 2. Define the StructuredGrid (Geometry ONLY)
modelgrid = fp.discretization.StructuredGrid(
    delr=m.dis.delr.array * 0.3048,
    delc=m.dis.delc.array * 0.3048,
    xoff=1795400 * 0.3048,
    yoff=1424400 * 0.3048,
    top=m.dis.top.array * 0.3048, # Use model top for geometry
    botm=m.dis.botm.array * 0.3048,
    idomain=idomain_layer1,
    crs="EPSG:3071"
)

# 3. Export the initial heads to GeoTIFF
# Use the modelgrid for spatial context, but ic_data for the values
fp.export.utils.export_array(
    modelgrid, 
    "lpr_ss_gwf.rcha_recharge_1.tif", 
    rcha_data, # This is where the data goes
    nodata=-999
)

/opt/conda/envs/hwrs564b/lib/python3.11/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
